# Tools

Contents:
- How to Build Tools
  - How Tool Calling Works
  - Building Hand-Crafted Tools
  - Wrapping Tools
  - Using Libraries
- Building Tools for Our Agent
  - Recap — Agent from Previous Lesson
  - Adding Real Tools


## 0. Environment setup

Install dependencies and create LLM wrapper

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-4o-mini"

client = OpenAI()

def call_llm(prompt: str) -> str:
    response = client.responses.create(
        model=MODEL,
        input=prompt,
    )

    return response.output_text

# 1. How to build tools

# 1.1. How tool-calling works

LLMs produce outputs as text; to affect the world (compute, fetch data, mutate state), the agent must call external tools.

In [ ]:
import json


def build_initial_prompt(tools_descr: str, user_text: str) -> str:
  system_prompt = f"""
      You are a tool-using assistant.

      You have access to the following tools:
      {tools_descr}

      When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
      {{"type":"tool_call","name":"<tool_name>","arguments":{{...}}}}

      When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
      {{"type":"final","answer":"..."}}
    """
  return f"{system_prompt}\n\nUSER: {user_text}\n"

def process_tool_call(json_model_output, tools_by_name):
    tool_name = json_model_output["name"]
    tool_args = json_model_output["arguments"]
    print(f'Executing tool {tool_name} with args: {tool_args}')

    if tool_name not in tools_by_name:
      raise ValueError("Unknown tool")

    tool_result = tools_by_name[tool_name](**tool_args)

    print(f'Tool result: {tool_result}')

    return tool_result


def run_agent_inner(prompt, tools_by_name):
    # call LLM
    model_output = call_llm(prompt)
    json_model_output = json.loads(model_output)
    print(f"model_output: {model_output}")

    # process tool-call if needed
    if json_model_output["type"] == "tool_call":
        tool_result = process_tool_call(json_model_output, tools_by_name)

        # call LLM with the tool_result
        prompt_with_tool_result = (
            f"{prompt}\n"
            f"ASSISTANT: {model_output}\n"
            f"TOOL_RESULT: {tool_result}\n"
        )
        return run_agent_inner(prompt_with_tool_result, tools_by_name)

    print("\n\n==================Last prompt==================")
    print(prompt)
    print(f"\n\n====Final result: {json_model_output['answer']}===")
    return json_model_output["answer"]



## 1.2. Building hand-crafted tools

In [ ]:
def multiply(a, b):
    return a * b

RAW_TOOLS_BY_NAME = {
    'multiply': multiply
}

RAW_TOOLS_DESC = """
        Tool name: multiply
        Description: Multiply two integers
        Arguments:
          - a: integer
          - b: integer
        Returns: integer
"""

def run_agent_with_raw_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(RAW_TOOLS_DESC, user_text)
    res = run_agent_inner(prompt, RAW_TOOLS_BY_NAME)

In [ ]:
run_agent_with_raw_tools("What is 17 multiplied by 23?")


In [ ]:
run_agent_with_raw_tools("What is 17 multiplied by 23 and multiplied by 10?")

In [ ]:
run_agent_with_raw_tools("What is the capital of Great Britain?")

Problems:

🔴 how to add lots of tools in the same format

🔴 what if something changes in the tool


## 1.3. Wrapping tools

### 1.3.1. Using class for unification

In [ ]:

from typing import Callable, Any, List, Tuple

class Tool:
    def __init__(
        self,
        name: str,
        description: str,
        func: Callable[..., Any],
        arguments: List[Tuple[str, str]],
        outputs: str,
    ):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        args_str = ", ".join([f"{n}: {t}" for n, t in self.arguments])
        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)


In [ ]:

def multiply(a, b):
    return a * b

class_multiply = Tool(
    "multiplier",
    "Multiply two integers",
    multiply,
    [("a", "int"), ("b", "int")],
    "int",
)

class_multiply.to_string()

Problems:

🟢 how to add lots of tools in the same format

🔴 what if something changes in the tool


### 1.3.2. "Decorating" tools

Decorator:
- Reads the function signature
- Extracts the argument types
- Extracts the return type
- Uses the docstring as the description


In [ ]:
import inspect

def wrap_tool(func: Callable[..., Any]) -> Tool:
    sig = inspect.signature(func)

    # args
    arguments = []
    for p in sig.parameters.values():
        ann = p.annotation
        if ann is inspect._empty:
            ann_name = "Any"
        else:
            ann_name = getattr(ann, "__name__", str(ann))

        arguments.append((p.name, ann_name))

    # return type
    ret = sig.return_annotation
    if ret is inspect._empty:
        outputs = "Any"
    else:
        outputs = getattr(ret, "__name__", str(ret))

    description = (func.__doc__ or "No description provided").strip()
    name = func.__name__

    return Tool(name, description, func, arguments, outputs)


In [ ]:
@wrap_tool
def multiply_tool(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

multiply_tool.to_string()

Problems:

🟢 how to add lots of tools in the same format

🟢 what if something changes in the tool


### 1.3.3. All together

In [ ]:
@wrap_tool
def multiply_tool(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

NICE_TOOLS_BY_NAME = {
    'multiply_tool': multiply_tool
}

NICE_TOOLS_DESC = "\n".join([t.to_string() for t in NICE_TOOLS_BY_NAME.values()])

def run_agent_with_nice_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(NICE_TOOLS_DESC, user_text)
    res = run_agent_inner(prompt, NICE_TOOLS_BY_NAME)

In [ ]:
run_agent_with_nice_tools("What is 17 multiplied by 23?")

## 1.4. Using libraries

So far we built tool usage **from scratch**:
- defined tool format
- wrote parser
- handled tool execution manually

Libraries like LangChain/LangGraph automate this process:
- tool registration
- tool calling protocol
- agent loop
- message formatting

We will rebuild the same example using LangChain.

In [ ]:
!pip install langgraph langchain langchain-openai

In [ ]:
from typing import Annotated, TypedDict, List

from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


# Agent state with a reducer: messages will be appended, not overwritten
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]


# Bind tools to the model so it can produce tool calls
llm = ChatOpenAI(model=MODEL, temperature=0).bind_tools([multiply])


# LLM node: produce either tool calls or a final answer
def llm_node(state: AgentState) -> dict:
    response = llm.invoke(state["messages"])
    # Return only the new message; the reducer will append it to state["messages"]
    return {"messages": [response]}


# Tool node: executes tool calls and returns ToolMessage(s)
tools_node = ToolNode([multiply])


# Build the graph
graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tools_node)

graph.set_entry_point("llm")

# Route to tools if the last AI message contains tool calls; otherwise end
graph.add_conditional_edges("llm", tools_condition, {"tools": "tools", END: END})

# After tools run, go back to the LLM so it can use tool results
graph.add_edge("tools", "llm")

app = graph.compile()
app

In [ ]:
result = app.invoke({
    "messages": [
        SystemMessage(content="You are a tool-using assistant. Always try to use provided tools."),
        HumanMessage(content="What is 17 multiplied by 23?")
    ]
})

print(result["messages"][-1].content)

In [ ]:
for step in app.stream(
    {
        "messages": [
          SystemMessage(content="You are a tool-using assistant. Always try to use provided tools."),
          HumanMessage(content="What is 17 multiplied by 23?")
        ]
    },
    stream_mode="values",
):
    last = step["messages"][-1]

    print("NODE OUTPUT:")
    print(type(last).__name__)

    if hasattr(last, "content"):
        print("Content:", last.content)

    if hasattr(last, "tool_calls") and last.tool_calls:
        print("Tool calls:", last.tool_calls)

    print("==========")

# 2. Building tools for our agent

## 2.1. Recap - agent from previous lesson


In [ ]:
!pip install -q langchain langgraph langchain-openai

In [ ]:
from typing import List, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, END


SYSTEM_PROMPT = """
  You are a virtual retail assistant.
  IMPORTANT: In this lesson you have NO tools, NO database access, NO RAG.
  You must be transparent about limitations:
  - You cannot look up products, reservations, user profiles, delivery on a specific reservation, insurance status, or real-time product status.
  - You can only: explain the policy, ask clarifying questions, and outline what info is needed to proceed.
  - Do NOT invent product details, reservation details, membership level, prices, or statuses.
  - Do NOT provide subjective recommendations.
"""

llm = ChatOpenAI(model=MODEL)

class AgentState(TypedDict):
    messages: List[BaseMessage]


def llm_node(state: AgentState) -> AgentState:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resp = llm.invoke(msgs)
    return {"messages": state["messages"] + [resp]}


graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.set_entry_point("llm")
graph.add_edge("llm", END)

app = graph.compile()

# --- tiny helper to talk to the agent ---
def ask(user_text: str, app) -> str:
    print("\n🟣 USER QUERY")
    print(">" * 50)
    print(user_text)

    state = {"messages": [HumanMessage(content=user_text)]}
    out = app.invoke(state)
    answer = out["messages"][-1].content

    print("\n🟢 AGENT RESPONSE")
    print("<" * 50)
    print(answer)
    print()


ask('Who are you?', app)

In [ ]:
app

In [ ]:
import json
from typing import List, TypedDict, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# ----------------------------
# 0) Tiny in-memory "database"
# NOTE: legacy keys are preserved for backward compatibility in lesson code.
# Domain semantics are now retail-oriented.
# ----------------------------
DATABASE = {
    "products": {
        # product_key -> catalog offer record
        "AY001_2026-02-22": {
            "product_key": "AY001_2026-02-22",
            "from": "HEL",
            "to": "AMS",
            "date": "2026-02-22",
            "time": "09:30",
            "seats": 3,
            "price": 120,
        },
        "AY101_2026-02-22": {
            "product_key": "AY101_2026-02-22",
            "from": "HEL",
            "to": "AMS",
            "date": "2026-02-22",
            "time": "16:10",
            "seats": 1,
            "price": 155,
        },
    },
    "orders": {
        # order_id -> order record
        "BKG-123": {
            "order_id": "BKG-123",
            "product_key": "AY001_2026-02-22",
            "customer_name": "Jane Doe",
            "status": "confirmed",
        }
    },
}


# -----------------------
# 1) Tools
# -----------------------
@tool
def get_order(order_id: str) -> str:
    """Retrieve a order by ID."""
    order_record = DATABASE["orders"].get(order_id)
    if not order_record:
        return json.dumps({"error": f"Order {order_id} not found"}, ensure_ascii=False)

    product_record = DATABASE["products"].get(order_record["product_key"], {})
    return json.dumps(
        {"order_id": order_id, **order_record, "product": product_record},
        ensure_ascii=False,
    )


@tool
def search_products(origin: str, destination: str, date: str) -> str:
    """Search available products."""
    matching_products = [
        product
        for product in DATABASE["products"].values()
        if product["from"] == origin
        and product["to"] == destination
        and product["date"] == date
        and product["seats"] > 0
    ]
    return json.dumps(
        sorted(matching_products, key=lambda f: f["time"]),
        ensure_ascii=False,
    )


@tool
def cancel_order(order_id: str) -> str:
    """Cancel a order."""
    order_record = DATABASE["orders"].get(order_id)
    if not order_record:
        return json.dumps({"error": "Order not found"}, ensure_ascii=False)

    product_record = DATABASE["products"].get(order_record["product_key"])
    order_record["status"] = "cancelled"

    if product_record:
        product_record["seats"] += 1

    refund_amount = product_record["price"] if product_record else 0
    return json.dumps({"success": True, "refund": refund_amount}, ensure_ascii=False)


TOOLS = [get_order, search_products, cancel_order]


# -----------------------
# 2) Prompt (system)
# -----------------------
POLICIES: List[str] = [
    "Use tools when they are needed to answer accurately",
    "Never invent order details, offer details, prices, or statuses",
    "If information is insufficient, ask clarifying questions.",
    "Request confirmation before any order cancellation. Ask explicitly and wait for user confirmation",
    "Retrieve the current order before proposing changes",
]

SYSTEM_PROMPT = ("You are a virtual retail support assistant." + f"\nPOLICIES: {chr(10).join(f"* {p}" for p in POLICIES)}").strip()


# -----------------------
# 3) Agent state with a reducer: messages will be appended, not overwritten
# -----------------------
class AgentStateRed(TypedDict):
    # messages: List[BaseMessage]
    messages: Annotated[List[BaseMessage], add_messages]


# -----------------------
# 4) LLM node
# -----------------------
# llm = ChatOpenAI(model=MODEL)
llm_with_tools = ChatOpenAI(model=MODEL).bind_tools(TOOLS)


# same
def llm_with_tools_node(state: AgentStateRed) -> AgentStateRed:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resp = llm_with_tools.invoke(msgs)
    return {"messages": [resp]}


# ----------------------------
# 5) Router: decide whether to call tools
# ----------------------------
def route_after_llm(state: AgentStateRed) -> Literal["tools", "end"]:
    last = state["messages"][-1]
    # If the model requested tool calls, send to ToolNode; else finish.
    has_tool_calls = getattr(last, "tool_calls", None)
    return "tools" if has_tool_calls else "end"


# ----------------------------
# 6) Build the graph
# ----------------------------
graph_with_tools = StateGraph(AgentStateRed)

graph_with_tools.add_node("llm", llm_with_tools_node)
graph_with_tools.add_node("tools", ToolNode(TOOLS))

graph_with_tools.set_entry_point("llm")

# If LLM asked for tools -> tools; otherwise -> END
graph_with_tools.add_conditional_edges(
    "llm",
    route_after_llm,
    {"tools": "tools", "end": END},
)

# After tools run, go back to LLM (so it can interpret tool results and answer / ask next question)
graph_with_tools.add_edge("tools", "llm")

app_with_tools = graph_with_tools.compile()

app_with_tools

In [ ]:
# --- demo (RetailOps phrasing over legacy tool names) ---
ask("Find catalog offers from HEL to AMS on 2026-02-22", app_with_tools)
ask("What's in order BKG-123?", app_with_tools)
ask("Cancel order BKG-123", app_with_tools)  # should ask for confirmation per policy (ideally)


In [ ]:
class Chat:
    def __init__(self, app):
        self.state = {"messages": []}
        self.app = app

    def ask(self, user_text: str) -> str:
        print("\n🟣 USER QUERY")
        print(">" * 50)
        print(user_text)

        self.state["messages"].append(HumanMessage(content=user_text))
        out = self.app.invoke(self.state)
        self.state = out
        answer = out["messages"][-1].content


        print("\n🟢 AGENT RESPONSE")
        print("<" * 50)
        print(answer)
        print()


chat = Chat(app_with_tools)
print(chat.ask("Cancel order BKG-123"))


In [ ]:
chat.ask("I confirm the cancellation of order BKG-123")